In [0]:
-- Databricks notebook source
-- ============================================================================
-- Bronze load: Happy Family Homestay bookings
--
-- Source : /Volumes/happy_family_homestay/default/homestay_landing/homestay_raw.csv
--          (landed by 01_bronze_ingestion.py from the Google Form responses)
-- Target : happy_family_homestay.bronze.homestay
-- Grain  : one row per form submission
-- Load   : incremental upsert (MERGE). Safe to re-run: the same submission is
--          matched on (timestamp, nama_lengkap, nomor_hp) and updated, not duplicated.
-- Note   : all columns are kept as STRING on purpose. Type casting and cleaning
--          happen in the Silver layer (02_silver_cleaning.py).
-- ============================================================================

-- COMMAND ----------

-- 1. Create the schema and the target table (no-op if they already exist)

CREATE SCHEMA IF NOT EXISTS happy_family_homestay.bronze
COMMENT 'Bronze layer: raw data as received from source systems';

CREATE TABLE IF NOT EXISTS happy_family_homestay.bronze.homestay (
    timestamp                                               STRING COMMENT 'Form submission time',
    nama_lengkap                                            STRING COMMENT 'Guest full name',
    nomor_hp                                                STRING COMMENT 'Guest mobile number',
    e_mail                                                  STRING COMMENT 'Guest email address',
    tanggal_check_in                                        STRING COMMENT 'Check-in date',
    tanggal_check_out                                       STRING COMMENT 'Check-out date',
    jumlah_tamu                                             STRING COMMENT 'Number of guests',
    jumlah_kamar                                            STRING COMMENT 'Number of rooms',
    kota                                                    STRING COMMENT 'Guest home city',
    tujuan_perjalanan_wisata                                STRING COMMENT 'Purpose of the trip',
    dari_mana_anda_mendengar_tentang_happy_family_homestay  STRING COMMENT 'How the guest heard about the homestay',
    total_bayar                                             STRING COMMENT 'Total amount paid'
)
USING DELTA
COMMENT 'Raw homestay booking confirmations from Google Form responses';

-- COMMAND ----------

-- 2. Read the landed CSV and rename the form headers to snake_case column names.
--    inferSchema is off, so every column is read as STRING (matches the table).

CREATE OR REPLACE TEMP VIEW tv_homestay AS
SELECT
    `Timestamp`                                               AS timestamp,
    `Nama Lengkap`                                            AS nama_lengkap,
    `Nomor HP`                                                AS nomor_hp,
    `E-mail`                                                  AS e_mail,
    `Tanggal Check-in`                                        AS tanggal_check_in,
    `Tanggal Check-out`                                       AS tanggal_check_out,
    `Jumlah Tamu`                                             AS jumlah_tamu,
    `Jumlah Kamar`                                            AS jumlah_kamar,
    `Kota`                                                    AS kota,
    `Tujuan Perjalanan/Wisata`                                AS tujuan_perjalanan_wisata,
    `Dari mana anda mendengar tentang Happy Family Homestay?` AS dari_mana_anda_mendengar_tentang_happy_family_homestay,
    `Total Bayar`                                             AS total_bayar
FROM read_files(
    '/Volumes/happy_family_homestay/default/homestay_landing/homestay_raw.csv',
    format      => 'csv',
    header      => true,
    inferSchema => false
);

-- COMMAND ----------

-- 3. Upsert into bronze.
--    A submission is identified by (timestamp, nama_lengkap, nomor_hp).
--    COALESCE makes NULLs compare as equal, so rows with a missing value still match.

MERGE INTO happy_family_homestay.bronze.homestay AS t
USING tv_homestay AS s
    ON  COALESCE(t.timestamp,    '') = COALESCE(s.timestamp,    '')
    AND COALESCE(t.nama_lengkap, '') = COALESCE(s.nama_lengkap, '')
    AND COALESCE(t.nomor_hp,     '') = COALESCE(s.nomor_hp,     '')
WHEN MATCHED THEN
    UPDATE SET *
WHEN NOT MATCHED THEN
    INSERT *;